In [0]:
%sql
use catalog de_workspace26;
use schema shopeasy_raw_arnav;

In [0]:
from pyspark.sql import functions as f


In [0]:
bronze_orders = spark.read.table("bronze_orders")
deduped = bronze_orders.dropDuplicates(["order_id"])

deduped.show(2)

In [0]:

deduped = deduped \
  .withColumn("order_date", f.to_date(f.col("order_date"), "yyyy-MM-dd")) \
  .withColumn("revenue", f.col("quantity") * f.col("unit_price"))

In [0]:
from pyspark.sql import Window
window_spec = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(Window.unboundedPreceding, 0)
deduped = deduped.withColumn("cumulative_revenue", f.sum("revenue").over(window_spec))

In [0]:
%sql
drop table if exists silver_orders;

In [0]:
customers = spark.read.table("bronze_customers")
products = spark.read.table("bronze_products")
customers.show(5)
products.show(5)
deduped.show(5)




enriched_data = deduped \
  .join(customers.select("customer_id", "city","loyalty_tier"), on="customer_id", how="left") \
  .join(products.select("product_id", "product_name", "category"), on="product_id", how="left")

enriched_data.show(5)

In [0]:
enriched_data.write.format('delta').mode('overwrite').partitionBy('region').saveAsTable('silver_orders')

In [0]:
spark.sql("""
  CREATE or replace TABLE silver_customers AS
  SELECT *, true AS is_current,
    current_date() AS effective_start_date,
    CAST(NULL AS DATE) AS effective_end_date
  FROM bronze_customers
""")

In [0]:
spark.sql("""
  MERGE INTO silver_customers AS target
  USING bronze_customers AS source
  ON target.customer_id = source.customer_id AND target.is_current = true
  WHEN MATCHED AND target.loyalty_tier != source.loyalty_tier THEN
    UPDATE SET target.is_current = false, target.effective_end_date = current_date()
  WHEN NOT MATCHED THEN
    INSERT (customer_id, name, email, city, loyalty_tier, signup_date, is_current, effective_start_date, effective_end_date)
    VALUES (source.customer_id, source.name, source.email, source.city, source.loyalty_tier, source.signup_date, true, current_date(), NULL)
""")

In [0]:
spark.sql("ALTER TABLE silver_orders SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
spark.sql("SELECT * FROM table_changes('silver_orders', 1)").show()